In [ ]:
from humancompatible.train.dual_optim import ALM, MoreauEnvelope
import torch
from torch import nn
import numpy as np

**Define regressor**

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, lsize):
        super().__init__()
        layers = []
        d = input_dim
        for dim in lsize:
            layers.append(nn.Linear(d, dim))
            layers.append(nn.Dropout(p=0.15))
            layers.append(nn.ReLU())
            d = dim

        layers.append(nn.Linear(d, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)
    
    def predict(self, X):
        return self.net(torch.tensor(X, dtype=torch.float32)).squeeze(-1)

**Prepare data**

Load config (used for solving DAG problem, setting target, etc)

In [ ]:
from omegaconf import OmegaConf

config_path = "experiments_conf/solver/hc_predictor.yaml"
cfg = OmegaConf.load(config_path)

problem_config_path = "experiments_conf/problem/codiet.yaml"
problem_cfg = OmegaConf.load(problem_config_path)

Load data

In [ ]:
import os
from data_helper import load_all_data

hydra_run_path = './mlruns/1/293ebf91b32e48999424332ac033350f'
# hydra_run_path = './mlruns/1/7ac7bdfaaea245c4bf5df1343155a37d'
with open(f"{hydra_run_path}/artifacts/selected_features.txt") as features:
    selected_features = features.read()[1:-1].split(', ')

target_col = problem_cfg.target
food_feats, non_food_feats, prep_data = load_all_data()     
prep_data = prep_data[selected_features + [target_col]]     # leave only selected and target columns

w_path = './outputs/2026-03-15/20-50-14/W_est.csv'
with open(w_path) as f:
    W = np.loadtxt(f, delimiter=",")

scafs: Dropping cols with many nans: ['scafs_aliquot-no', 'scafs_volume-per-aliqote', 'scafs_day', 'scafs_age', 'scafs_gender', 'scafs_box-position']. 
scafs: (282, 13)
ms_urine: (300, 36)
ms_serum: Dropping cols with many nans: ['ms_serum_aminoadipic-acid', 'ms_serum_beta-alanine', 'ms_serum_gamma-aminobutyric-acid']. 
ms_serum: (300, 33)
nmr_urine: (298, 53)
ms_lip: (301, 46)
dbs_rbc_lip: (299, 60)
microbiome: prep shape: (151, 4)
scafs: prep shape: (149, 5)
ms_serum: prep shape: (154, 30)
ms_urine: prep shape: (154, 33)
nmr_urine: prep shape: (153, 50)
lipidomics: prep shape: (155, 45)
lipidomics_dbs_rbc: prep shape: (153, 59)
microbiome_4_cl: prep shape: (151, 5)
microbiome_phyl4_cl: prep shape: (151, 5)
microbiome_embedding: prep shape: (151, 21)
microbiome_clean15: prep shape: (151, 16)

Total prep_data shape: (140, 488)
0      1.0
1      1.0
2      1.0
3      1.0
4      0.0
      ... 
135    0.0
136    1.0
137    1.0
138    1.0
139    1.0
Name: gender_numeric, Length: 140, dtype

In [ ]:
prep_data = prep_data.dropna(subset=[target_col])
prep_data = prep_data.dropna(axis=1)
X = prep_data.drop(target_col, axis=1) if target_col in prep_data.columns else prep_data
y = prep_data[target_col]
X = X.to_numpy()
y = y.to_numpy()

Train:

In [ ]:
# network params
lsize = [32, 16]

# training method params
lr = 0.25
init_lambda = 1.
lambda_lr = 1.0
penalty = 0.

sample_weight = None

# training process params
n_outer = 10
n_inner = 100
verbose = True

torch.manual_seed(42)

In [ ]:
import networkx as nx
import solve_milp

def W_constraint(M, muX, yhat):
    muY = yhat.mean()
    xy_bar = torch.concat([muX, muY.unsqueeze(0)])
    g = torch.abs(xy_bar - M @ xy_bar)
    return g

def fit_nn_estimator(X, y, W, model, verbose=False, constrained = True):
    
    n, d = X.shape

    assert W.shape == (d + 1, d + 1), \
    "W must be (d+1)x(d+1)"
    
    base_opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.25)
    optimizer = MoreauEnvelope(base_opt) if constrained else base_opt
    dual_opt = ALM(m=d+1, lr=lambda_lr, penalty=penalty, init_duals=init_lambda)

    # ------------------------------
    # Precompute parts of constraint
    # ------------------------------
    M = W - torch.eye(d + 1)
    muX = X.mean(dim=0)
    loss = torch.nn.MSELoss()

    for outer in range(n_outer):
        for inner in range(n_inner):
            optimizer.zero_grad()
            yhat = model(X)
            mse = loss(yhat, y)

            if constrained:
                g = W_constraint(M, muX, yhat)
                aug_loss = dual_opt.forward(loss=mse, constraints=g) 
                aug_loss.backward()
            else:
                mse.backward()

            optimizer.step()

        if constrained:
            with torch.no_grad():
                muY = model(X).mean()
                xy_bar = torch.concat([muX, muY.unsqueeze(0)])
                g = torch.abs(xy_bar - M @ xy_bar)
                dual_opt.update(g)
            lam = dual_opt.duals.detach().numpy()

        for param_group in optimizer.param_groups:
            param_group['lr'] *= 0.95

        if verbose:
            print(
            f"outer={outer:02d}  "
            f"mean(yhat)={muY.item():+.6e}  "
            f"MSE={mse.item():+.6e}  "
            f"||g||={np.linalg.norm(g).item():.6e}  "
            f"lambda_norm={np.linalg.norm(lam).item():.6e}  "
        )


def get_current_column_names(X):
        current_feature_names = []

        for i in range(X.shape[1]):
            col_data = X[:, i]
            for col_name in prep_data.columns:
                if col_name in current_feature_names:
                    continue
                it = iter(prep_data[col_name])
                if all(any(a == b for a in it) for b in col_data):
                    current_feature_names.append(col_name)
                    break

        return current_feature_names

def calculate_dag(X, y):
    d = X.shape[1] + 1 # adding one for y
    X_y = np.column_stack((X, y))
    current_column_names = get_current_column_names(X)
    G = nx.read_graphml(os.path.join(cfg.data_path, cfg.knowledge_graph_filename))
    H = G.subgraph(current_column_names + [target_col]).copy()
    if H.number_of_nodes() > 0:
        print('not emty')
    H = nx.complement(H)
    col_to_idx = {col: idx for idx, col in enumerate(current_column_names + [target_col])}
    tabu_edges = list((col_to_idx[s],col_to_idx[e]) for (s,e) in H.edges())
    w_est, _, _, _, _ = solve_milp.solve(X_y, cfg, cfg.nonzero_threshold,
                                                            Y=[],
                                                            B_ref=np.zeros((d,d)),
                                                            tabu_edges=tabu_edges )
    return w_est

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.base import BaseEstimator
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

loss = torch.nn.MSELoss()

class ScikitModelWrapper(BaseEstimator):
    def __init__(self, W, constrained, recalculate_dag):
        self.W = W
        self.scaler = StandardScaler()
        self.recalculate_dag = recalculate_dag
        self.constrained = constrained
        self.is_fitted_ = False
    

    def fit(self, X, y):
        # calculate dag
        if self.recalculate_dag:
            self.W = calculate_dag(X, y)
        elif self.W is None:
            raise ValueError(f'recalculate_dag set to {self.recalculate_dag} but W not provided!')
        # create model
        self.model = MLPRegressor(input_dim=X.shape[1], lsize=lsize)
        # data to tensor
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        self.W = torch.tensor(self.W, dtype=torch.float32)
        fit_nn_estimator(X, y, self.W, self.model, constrained=self.constrained)
        self.is_fitted_ = True
        return self
    
    @torch.inference_mode
    def predict(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        return self.model(X).detach().numpy()

@torch.inference_mode
def compute_predictor_errors_scikit(estimator, X, y):
    test_mse = mean_squared_error(y, estimator.predict(X))
    return test_mse 


pipe = make_pipeline(StandardScaler(), ScikitModelWrapper(W, constrained=True, recalculate_dag=True))

results = cross_validate(
    pipe,
    X,
    y,
    cv = 10,
    scoring=compute_predictor_errors_scikit,
    return_train_score=True
)

not emty
not emty
not emty
not emty
not emty
not emty
not emty
not emty
not emty
not emty


In [ ]:
results['train_score'].mean() #/ score_normalizer

np.float64(5.303110987119567)

In [ ]:
results['test_score'].mean() #/ score_normalizer

np.float64(6.149605694835186)

Try XGBoost on same data:

In [ ]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline


n_estimators = 10
max_depth = 3
learning_rate = 0.1
random_state = 42


model_class = Pipeline
model_params = {
    'steps': [
        ("scale", StandardScaler()),
        ("xgb", XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            random_state=random_state,
        )
        )
    ]
}

xgboost_model = model_class(**model_params)

In [ ]:
# from sklearn.model_selection import GridSearchCV


# gridsearch = GridSearchCV(
#     estimator=Pipeline((('scaler', StandardScaler()), ('xgb', XGBRegressor()))),
#     param_grid={'xgb__n_estimators': [3, 5, 10, 15, 20], 'xgb__max_depth': [1, 2, 3, 4, 5], 'xgb__learning_rate': [0.15, 0.2, 0.25]},
#     scoring=lambda e, X, y: -1*compute_predictor_errors_scikit(e, X, y),
#     return_train_score=True,
#     cv=10,
# )

# gridsearch.fit(X, y)
# gridsearch.best_score_

In [ ]:
results = cross_validate(
    xgboost_model,
    X,
    y,
    cv = 10,
    scoring=compute_predictor_errors_scikit,
    return_train_score=True,
)

In [ ]:
results['train_score'].mean() #/ score_normalizer

np.float64(3.9003100049237247)

In [ ]:
results['test_score'].mean() #/ score_normalizer

np.float64(6.689691950611879)